# 第3章　データセットの構築とキュレーション ― AI開発の基盤を作る

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 匿名化を手順に落とす ― DICOMタグと検証

In [ ]:
import pydicom
import datetime, hashlib
from pydicom.uid import generate_uid

def compute_age(birth_da, study_da):
    """DICOM の DA 値（YYYYMMDD）2つから満年齢を返す。形式が不正なら ValueError で止まる。"""
    b = datetime.datetime.strptime(str(birth_da), "%Y%m%d").date()
    s = datetime.datetime.strptime(str(study_da), "%Y%m%d").date()
    return s.year - b.year - ((s.month, s.day) < (b.month, b.day))

def patient_offset_days(patient_key, salt, max_days=365):
    """患者ごとに固定した日付の偏移（±max_days 日）。鍵付きハッシュで決めるので、
    同じ患者は何度処理しても同じ偏移になる。既知の日付等から偏移量が推定される可能性は残る。"""
    h = hashlib.sha256((salt + "|" + str(patient_key)).encode("utf-8")).digest()
    return int.from_bytes(h[:4], "big") % (2 * max_days + 1) - max_days

def shift_date(da, offset_days):
    """DA 値を偏移させ、YYYYMMDD の形式のまま返す（「基準日からの日数」のような値を日付の項目に入れない）。"""
    d = datetime.datetime.strptime(str(da), "%Y%m%d").date() + datetime.timedelta(days=offset_days)
    return d.strftime("%Y%m%d")

UID_ROOT = "1.2.826.0.1.3680043.8.498."   # 例として pydicom のルート。実運用では自組織に割り当てられた UID ルートを使う

def remap_uid(uid, salt, root=UID_ROOT):
    """元の UID と salt から決定論的に新しい UID を作る（同じ入力 → 同じ出力なので、同一検査の紐付けが保てる）。
    pydicom の generate_uid は、ルート（prefix）と entropy_srcs から SHA-512 で数値部を作り、64文字以内に収めた
    正しい形式の UID を返す。prefix=None にすると entropy_srcs が無視されて毎回ランダムになるので、必ずルートを渡す。"""
    return generate_uid(prefix=root, entropy_srcs=[str(uid), salt])

In [ ]:
import pydicom
SALT = "運用時は鍵管理の仕組みで保管する秘密の文字列"   # データや原稿と一緒に置かない。鍵が漏れれば偏移もUIDも逆算されうる
ds = pydicom.dcmread("in.dcm")
offset = patient_offset_days(ds.get("PatientID", ""), SALT)   # 識別子を消す前に、患者ごとの偏移を決めておく
for tag in ["PatientName","PatientID","OtherPatientIDs","ReferringPhysicianName",
            "InstitutionName","OperatorsName","StudyDescription"]:
    if tag in ds: setattr(ds, tag, "")
if ds.get("PatientBirthDate") and ds.get("StudyDate"):  # 無い症例に加え、タグはあるが値が空の症例もある。値の有無で判定する
    age = compute_age(ds.PatientBirthDate, ds.StudyDate)   # 偏移させる前の日付で、撮影日 − 生年月日 の満年齢を出す
    ds.PatientAge = f"{age:03d}Y"                      # 生年月日は年齢へ丸める
ds.PatientBirthDate = ""                            # タグは残して空値化し、年齢計算を省いた場合も元の生年月日を残さない
# 日付は患者ごとに一貫してシフトし、有効な日付形式を維持。受付番号は除去
for tag in ["StudyDate", "SeriesDate", "AcquisitionDate", "ContentDate"]:
    if ds.get(tag):                                 # 空値（Type 2 の未記入）は strptime が落ちるので触らない
        setattr(ds, tag, shift_date(getattr(ds, tag), offset))
ds.AccessionNumber = ""
# UIDは辞書で一貫再発番（同一検査の紐付けを維持しつつ匿名化）。
# Studyだけ変えても、SeriesとSOPが残れば元PACSの当該画像に逆引きできてしまう。
for tag in ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID"]:
    setattr(ds, tag, remap_uid(getattr(ds, tag), SALT))
ds.file_meta.MediaStorageSOPInstanceUID = ds.SOPInstanceUID   # file_meta側も揃える
ds.remove_private_tags()                       # プライベートタグは原則除去
ds.save_as("out.dcm")

## データ品質を、継続的に検査する ― 取り込み時の関所

In [ ]:
# 取り込み時のデータ品質検査（期待仕様に照らして合否を出す）
def check(study, index):
    ok  = study.modality in {"CT", "MR"}                    # 想定モダリティ
    ok &= 0.3 <= study.pixel_spacing_mm <= 1.5              # 画素間隔が現実的範囲（数値は例示）
    ok &= study.slices >= 20                                # スライス枚数の下限
    ok &= not phi_present(study.tags)                       # 匿名化の取りこぼし検査
    if study.modality == "CT":                              # 値域の検査はモダリティで分岐する
        ok &= study.hu_converted                            # HUへ変換済み（変換係数あり）か
        ok &= (-1100 <= study.hu_min) and (study.hu_max <= 3100)   # プロトコル別のHU基準（例示）
    else:                                                   # MRにHUは無い。シーケンスに応じた信号強度・品質基準へ
        ok &= mr_intensity_ok(study)
    ok &= not is_near_duplicate(study, index)              # 重複・近重複でないか
    return ok
# 不合格は破棄せず「隔離」し、理由付きで記録（後で見直せるように）